# Benchmark de Modelos Leves para Detecção de Phishing

**Dataset:** `kmack/Phishing_urls` (HuggingFace) — 708k URLs

**Modelos avaliados:**
- Logistic Regression
- Linear SVM
- Random Forest
- Gradient Boosting (HistGradientBoosting)

**Métricas:**
Accuracy, Precision, Recall, F1-Score, AUC-ROC, PR-AUC, MCC, Log Loss, Specificity, Latência P50/P95/P99, Tempo de treino

In [ ]:
# ============================================================
# 1. Instalação e Imports
# ============================================================
!pip install -q scikit-learn pandas numpy matplotlib seaborn datasets tldextract

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import re
import warnings
warnings.filterwarnings('ignore')

from urllib.parse import urlparse
import tldextract

from datasets import load_dataset

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, auc,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, log_loss, matthews_corrcoef,
    precision_recall_curve, average_precision_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

print("Imports OK.")

In [ ]:
# ============================================================
# 2. Carregar Dataset do HuggingFace
# ============================================================
print("Carregando dataset kmack/Phishing_urls...")

dataset = load_dataset("kmack/Phishing_urls")
df = pd.concat([
    dataset['train'].to_pandas(),
    dataset['test'].to_pandas(),
    dataset['valid'].to_pandas()
], ignore_index=True)

print(f"Total: {len(df)} URLs")
print(f"\nDistribuição:")
print(df['label'].value_counts())
print(f"\nExemplos:")
df.head(10)

In [ ]:
# ============================================================
# 3. Feature Engineering
# ============================================================
# Como o dataset tem apenas a URL (texto), precisamos extrair
# features numéricas para os modelos de ML clássico.
# ============================================================

def extract_features(url: str) -> dict:
    """Extrai features numéricas de uma URL."""
    # Garantir que a URL tenha esquema para o urlparse funcionar
    raw = url
    if not url.startswith(('http://', 'https://')):
        url = 'http://' + url

    try:
        parsed = urlparse(url)
    except Exception:
        parsed = urlparse('http://invalid')

    try:
        ext = tldextract.extract(url)
    except Exception:
        ext = tldextract.extract('http://invalid')

    hostname = parsed.hostname or ''
    path = parsed.path or ''
    query = parsed.query or ''

    features = {}

    # --- Comprimento ---
    features['url_length'] = len(raw)
    features['hostname_length'] = len(hostname)
    features['path_length'] = len(path)
    features['query_length'] = len(query)

    # --- Contagens de caracteres ---
    features['num_dots'] = raw.count('.')
    features['num_hyphens'] = raw.count('-')
    features['num_underscores'] = raw.count('_')
    features['num_slashes'] = raw.count('/')
    features['num_ats'] = raw.count('@')
    features['num_ampersands'] = raw.count('&')
    features['num_equals'] = raw.count('=')
    features['num_digits'] = sum(c.isdigit() for c in raw)
    features['num_special'] = sum(not c.isalnum() and c not in './-_' for c in raw)
    features['num_params'] = query.count('=') if query else 0

    # --- Proporções ---
    features['digit_ratio'] = features['num_digits'] / max(len(raw), 1)
    features['special_ratio'] = features['num_special'] / max(len(raw), 1)

    # --- Profundidade do path ---
    features['path_depth'] = path.count('/') - 1 if path else 0

    # --- Domínio ---
    features['subdomain_length'] = len(ext.subdomain)
    features['domain_length'] = len(ext.domain)
    features['tld_length'] = len(ext.suffix)
    features['num_subdomains'] = ext.subdomain.count('.') + 1 if ext.subdomain else 0

    # --- Presença de padrões suspeitos ---
    features['has_ip'] = int(bool(re.match(
        r'^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$', hostname
    )))
    features['has_at_sign'] = int('@' in raw)
    features['has_double_slash_redirect'] = int('//' in raw[8:])  # ignora o ://
    features['has_hex_encoding'] = int('%' in raw)
    features['has_https'] = int(raw.startswith('https'))
    features['has_www'] = int('www.' in raw.lower())
    try:
        port = parsed.port
    except ValueError:
        port = None
    features['has_port'] = int(port is not None and port not in [80, 443])

    # --- Entropia da URL (Shannon) ---
    if len(raw) > 0:
        prob = [raw.count(c) / len(raw) for c in set(raw)]
        features['entropy'] = -sum(p * np.log2(p) for p in prob if p > 0)
    else:
        features['entropy'] = 0

    # --- Extensão do arquivo ---
    suspicious_ext = ['.exe', '.zip', '.rar', '.js', '.php', '.html', '.htm',
                      '.cgi', '.asp', '.aspx', '.scr', '.bat', '.cmd']
    features['has_suspicious_ext'] = int(any(path.lower().endswith(e) for e in suspicious_ext))

    # --- Palavras suspeitas ---
    phishing_keywords = ['login', 'signin', 'verify', 'account', 'update',
                         'secure', 'banking', 'confirm', 'password', 'suspend',
                         'alert', 'paypal', 'ebay', 'amazon', 'apple', 'microsoft']
    raw_lower = raw.lower()
    features['num_phishing_keywords'] = sum(kw in raw_lower for kw in phishing_keywords)

    # --- Token count ---
    tokens = re.split(r'[.\-_/=?&]', raw)
    features['num_tokens'] = len(tokens)
    features['avg_token_length'] = np.mean([len(t) for t in tokens if t]) if tokens else 0
    features['max_token_length'] = max((len(t) for t in tokens if t), default=0)

    return features


print("Extraindo features das URLs... (pode levar alguns minutos)")
t0 = time.time()

feature_rows = df['text'].apply(extract_features)
df_features = pd.DataFrame(feature_rows.tolist())

elapsed = time.time() - t0
print(f"Features extraídas em {elapsed:.1f}s")
print(f"Shape: {df_features.shape} ({df_features.shape[1]} features)")
print(f"\nFeatures: {list(df_features.columns)}")
df_features.head()

In [ ]:
# ============================================================
# 4. Pré-processamento e Split
# ============================================================
RANDOM_STATE = 42
TEST_SIZE = 0.2
CV_FOLDS = 5

X = df_features.copy()
y = df['label'].astype(int)

# Tratar infinitos e NaN
X = X.replace([np.inf, -np.inf], np.nan)
if X.isnull().sum().sum() > 0:
    print(f"Valores nulos: {X.isnull().sum().sum()}. Preenchendo com mediana.")
    X = X.fillna(X.median())

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Train: {X_train.shape[0]} amostras | Test: {X_test.shape[0]} amostras")
print(f"Features: {X_train.shape[1]}")
print(f"Train - Phishing: {y_train.sum()} ({y_train.mean():.1%}) | Legítimo: {(~y_train.astype(bool)).sum()}")
print(f"Test  - Phishing: {y_test.sum()} ({y_test.mean():.1%}) | Legítimo: {(~y_test.astype(bool)).sum()}")

In [ ]:
# ============================================================
# 5. Definição dos Modelos
# ============================================================
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE,
            solver="lbfgs"
        ))
    ]),

    "Linear SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", CalibratedClassifierCV(
            estimator=LinearSVC(
                max_iter=2000,
                random_state=RANDOM_STATE
            ),
            cv=3
        ))
    ]),

    "Random Forest": Pipeline([
        ("clf", RandomForestClassifier(
            n_estimators=200,
            max_depth=20,
            min_samples_split=5,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),

    "Gradient Boosting": Pipeline([
        ("clf", HistGradientBoostingClassifier(
            max_iter=200,
            max_depth=6,
            learning_rate=0.1,
            random_state=RANDOM_STATE
        ))
    ]),
}

print(f"{len(models)} modelos definidos: {list(models.keys())}")

In [ ]:
# ============================================================
# 6. Treino e Avaliação
# ============================================================
results = []
trained_models = {}
predictions = {}

for name, pipeline in models.items():
    print(f"\n{'='*60}")
    print(f"Treinando: {name}")
    print(f"{'='*60}")

    # --- Treino ---
    t_start = time.time()
    pipeline.fit(X_train, y_train)
    train_time = time.time() - t_start

    # --- Predições ---
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    # --- Latência de inferência (por amostra) ---
    latencies = []
    for i in range(min(500, len(X_test))):
        sample = X_test.iloc[[i]]
        t0 = time.perf_counter()
        pipeline.predict(sample)
        latencies.append((time.perf_counter() - t0) * 1000)  # ms
    latencies = np.array(latencies)

    # --- Métricas ---
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc_roc = roc_auc_score(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    mcc = matthews_corrcoef(y_test, y_pred)
    logloss = log_loss(y_test, y_proba)

    # Especificidade (TNR)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp)

    row = {
        "Modelo": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "AUC-ROC": auc_roc,
        "Avg Precision (PR-AUC)": ap,
        "MCC": mcc,
        "Log Loss": logloss,
        "Specificity": specificity,
        "Latência P50 (ms)": np.percentile(latencies, 50),
        "Latência P95 (ms)": np.percentile(latencies, 95),
        "Latência P99 (ms)": np.percentile(latencies, 99),
        "Tempo Treino (s)": train_time,
    }
    results.append(row)

    trained_models[name] = pipeline
    predictions[name] = {"y_pred": y_pred, "y_proba": y_proba}

    print(f"  Accuracy:  {acc:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  AUC-ROC:   {auc_roc:.4f}")
    print(f"  MCC:       {mcc:.4f}")
    print(f"  Treino:    {train_time:.2f}s")
    print(f"  Latência:  P50={np.percentile(latencies, 50):.2f}ms | P95={np.percentile(latencies, 95):.2f}ms")

print("\nTodos os modelos treinados e avaliados!")

In [ ]:
# ============================================================
# 7. Tabela Comparativa
# ============================================================
df_results = pd.DataFrame(results).set_index("Modelo")

def highlight_best(s):
    """Destaca o melhor valor em cada coluna."""
    if s.name == "Log Loss" or "Latência" in s.name or "Tempo" in s.name:
        is_best = s == s.min()
    else:
        is_best = s == s.max()
    return ["background-color: #c6efce; font-weight: bold" if v else "" for v in is_best]

styled = (
    df_results.style
    .apply(highlight_best, axis=0)
    .format({
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-Score": "{:.4f}",
        "AUC-ROC": "{:.4f}",
        "Avg Precision (PR-AUC)": "{:.4f}",
        "MCC": "{:.4f}",
        "Log Loss": "{:.4f}",
        "Specificity": "{:.4f}",
        "Latência P50 (ms)": "{:.2f}",
        "Latência P95 (ms)": "{:.2f}",
        "Latência P99 (ms)": "{:.2f}",
        "Tempo Treino (s)": "{:.2f}",
    })
    .set_caption("Comparação de Modelos Leves - Detecção de Phishing (kmack/Phishing_urls)")
)

display(styled)

print("\n" + "="*80)
print("TABELA COMPARATIVA (texto)")
print("="*80)
print(df_results.round(4).to_string())

In [ ]:
# ============================================================
# 8. Matrizes de Confusão
# ============================================================
fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4.5))

for ax, (name, preds) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, preds["y_pred"])
    disp = ConfusionMatrixDisplay(cm, display_labels=["Legítimo", "Phishing"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(name, fontsize=13, fontweight="bold")

plt.suptitle("Matrizes de Confusão", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("matrizes_confusao.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvo: matrizes_confusao.png")

In [ ]:
# ============================================================
# 9. Curvas ROC e Precision-Recall
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

# --- Curva ROC ---
ax = axes[0]
for (name, preds), color in zip(predictions.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, preds["y_proba"])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC = {roc_auc:.4f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Random (AUC = 0.5)")
ax.set_xlabel("Taxa de Falso Positivo (FPR)", fontsize=12)
ax.set_ylabel("Taxa de Verdadeiro Positivo (TPR)", fontsize=12)
ax.set_title("Curvas ROC", fontsize=14, fontweight="bold")
ax.legend(loc="lower right", fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])

# --- Curva Precision-Recall ---
ax = axes[1]
for (name, preds), color in zip(predictions.items(), colors):
    prec_vals, rec_vals, _ = precision_recall_curve(y_test, preds["y_proba"])
    ap_val = average_precision_score(y_test, preds["y_proba"])
    ax.plot(rec_vals, prec_vals, color=color, lw=2, label=f"{name} (AP = {ap_val:.4f})")

ax.set_xlabel("Recall", fontsize=12)
ax.set_ylabel("Precision", fontsize=12)
ax.set_title("Curvas Precision-Recall", fontsize=14, fontweight="bold")
ax.legend(loc="lower left", fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])

plt.tight_layout()
plt.savefig("curvas_roc_pr.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvo: curvas_roc_pr.png")

In [ ]:
# ============================================================
# 10. Curvas ROC individuais por modelo
# ============================================================
fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4.5))

for ax, ((name, preds), color) in zip(axes, zip(predictions.items(), colors)):
    fpr, tpr, _ = roc_curve(y_test, preds["y_proba"])
    roc_auc = auc(fpr, tpr)

    ax.plot(fpr, tpr, color=color, lw=2)
    ax.fill_between(fpr, tpr, alpha=0.15, color=color)
    ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.4)
    ax.set_title(f"{name}\nAUC = {roc_auc:.4f}", fontsize=12, fontweight="bold")
    ax.set_xlabel("FPR")
    ax.set_ylabel("TPR")
    ax.grid(True, alpha=0.3)

plt.suptitle("Curvas ROC Individuais", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("curvas_roc_individuais.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvo: curvas_roc_individuais.png")

In [ ]:
# ============================================================
# 11. Gráfico de Barras Comparativo
# ============================================================
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1-Score", "AUC-ROC", "MCC"]
df_plot = df_results[metrics_to_plot]

fig, ax = plt.subplots(figsize=(14, 6))
df_plot.plot(kind="bar", ax=ax, width=0.75, edgecolor="black", linewidth=0.5)

ax.set_ylabel("Score", fontsize=12)
ax.set_title("Comparação de Métricas por Modelo", fontsize=14, fontweight="bold")
ax.set_xticklabels(df_plot.index, rotation=0, fontsize=11)
ax.set_ylim(0, 1.05)
ax.legend(loc="lower right", fontsize=9, ncol=2)
ax.grid(axis="y", alpha=0.3)

for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", fontsize=7, padding=2)

plt.tight_layout()
plt.savefig("comparacao_metricas.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvo: comparacao_metricas.png")

In [ ]:
# ============================================================
# 12. Comparação de Latência
# ============================================================
latency_cols = ["Latência P50 (ms)", "Latência P95 (ms)", "Latência P99 (ms)"]
df_lat = df_results[latency_cols]

fig, ax = plt.subplots(figsize=(10, 5))
df_lat.plot(kind="bar", ax=ax, width=0.7, edgecolor="black", linewidth=0.5,
            color=["#66b3ff", "#ff9999", "#ff6666"])

ax.set_ylabel("Latência (ms)", fontsize=12)
ax.set_title("Latência de Inferência por Modelo (single sample)", fontsize=14, fontweight="bold")
ax.set_xticklabels(df_lat.index, rotation=0, fontsize=11)
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", fontsize=8, padding=2)

plt.tight_layout()
plt.savefig("comparacao_latencia.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvo: comparacao_latencia.png")

In [ ]:
# ============================================================
# 13. Cross-Validation (5-Fold)
# ============================================================
print(f"Executando Cross-Validation ({CV_FOLDS}-Fold)...")
print("(Isso pode levar vários minutos com 708k amostras)\n")

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
cv_results = []

scoring = {
    "accuracy": "accuracy",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc",
}

for name, pipeline in models.items():
    print(f"  CV: {name}...", end=" ")
    t0 = time.time()
    scores = cross_validate(
        pipeline, X, y, cv=cv, scoring=scoring, n_jobs=-1
    )
    row = {"Modelo": name}
    for metric in scoring:
        vals = scores[f"test_{metric}"]
        row[f"{metric} (mean)"] = vals.mean()
        row[f"{metric} (std)"] = vals.std()
    cv_results.append(row)
    print(f"{time.time()-t0:.1f}s | Acc={row['accuracy (mean)']:.4f} | F1={row['f1 (mean)']:.4f} | AUC={row['roc_auc (mean)']:.4f}")

df_cv = pd.DataFrame(cv_results).set_index("Modelo")

cv_display = pd.DataFrame(index=df_cv.index)
for metric in scoring:
    cv_display[metric] = df_cv.apply(
        lambda r: f"{r[f'{metric} (mean)']:.4f} \u00b1 {r[f'{metric} (std)']:.4f}", axis=1
    )

print("\n" + "="*80)
print(f"CROSS-VALIDATION ({CV_FOLDS}-Fold) \u2014 mean \u00b1 std")
print("="*80)
display(cv_display)

In [ ]:
# ============================================================
# 14. Feature Importance (Random Forest + Gradient Boosting)
# ============================================================
from sklearn.inspection import permutation_importance

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, model_name in zip(axes, ["Random Forest", "Gradient Boosting"]):
    pipe = trained_models[model_name]
    clf = pipe.named_steps["clf"]

    if hasattr(clf, "feature_importances_"):
        importances = clf.feature_importances_
    else:
        # Fallback: permutation importance
        print(f"  {model_name}: usando permutation importance (pode levar um pouco)...")
        perm = permutation_importance(pipe, X_test, y_test, n_repeats=5, random_state=42, n_jobs=-1)
        importances = perm.importances_mean

    feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=True)
    top15 = feat_imp.tail(15)

    top15.plot(kind="barh", ax=ax, color="steelblue", edgecolor="black", linewidth=0.5)
    ax.set_title(f"{model_name} - Top 15 Features", fontsize=13, fontweight="bold")
    ax.set_xlabel("Importância")

plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvo: feature_importance.png")

In [ ]:
# ============================================================
# 15. Exportar resultados
# ============================================================
df_results.to_csv("resultados_modelos_leves.csv")
df_cv.to_csv("resultados_cross_validation.csv")
df_features.to_csv("features_extraidas.csv", index=False)

print("Arquivos exportados:")
print("  - resultados_modelos_leves.csv")
print("  - resultados_cross_validation.csv")
print("  - features_extraidas.csv")
print("  - matrizes_confusao.png")
print("  - curvas_roc_pr.png")
print("  - curvas_roc_individuais.png")
print("  - comparacao_metricas.png")
print("  - comparacao_latencia.png")
print("  - feature_importance.png")
print("\nDone!")